# L16: ReAct 模式

> 理解 Agent 的核心推理模式：推理 + 行动

In [ ]:
# === 环境设置 ===
import sys
import os
import json
import asyncio
from pathlib import Path
from typing import List, Dict, Any, Optional

# 自动找到项目根目录
current_path = Path(os.getcwd())
project_path = None

for path in [current_path] + list(current_path.parents):
    if (path / "backend").exists():
        project_path = str(path)
        break

if project_path and project_path not in sys.path:
    sys.path.insert(0, project_path)

# 加载 .env
env_file = Path(project_path) / ".env" if project_path else None
if env_file and env_file.exists():
    with open(env_file, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line and not line.startswith("#") and "=" in line:
                key, value = line.split("=", 1)
                os.environ[key.strip()] = value.strip()

print(f"项目路径: {project_path}")
print(f"Python 版本: {sys.version.split()[0]}")

from backend.agents.react_agent import ReActAgent
from backend.llm.models import Message
print("✓ 模块导入成功")

## 16.1 什么是 ReAct 模式？

### ReAct = Reasoning + Acting

**ReAct** 是一种让 LLM 交替进行**推理**和**行动**的模式。

```
传统 Chain-of-Thought:
  问题 → 思考 → 思考 → 思考 → 答案
  
ReAct 模式:
  问题 → 思考 → 行动 → 观察 → 思考 → 行动 → 观察 → ... → 答案
```

### 核心组件

| 组件 | 说明 | 示例 |
|------|------|------|
| **Thought** | 推理思考 | "我需要查询天气信息" |
| **Action** | 执行动作 | 调用 get_weather 工具 |
| **Observation** | 观察结果 | "今天北京晴天，22°C" |

### ReAct 的优势

```
1. 推理透明：可以看到 Agent 的思考过程
2. 纠错能力：观察结果后可以调整策略
3. 工具使用：可以调用外部工具获取信息
4. 多步推理：处理复杂任务需要多步
```

## 16.2 查看项目中的 ReAct 实现

In [ ]:
import inspect
from backend.agents.react_agent import ReActAgent

print("=== ReActAgent 源码（部分）===\n")
source = inspect.getsource(ReActAgent)
print(source[:2000] + "\n... (源码已截断)")

## 16.3 ReAct 的工作流程

### 完整循环

```
┌─────────────────────────────────────────────────────┐
│                   用户问题                           │
└─────────────────────────────────────────────────────┘
                         ↓
┌─────────────────────────────────────────────────────┐
│  Thought 1: 分析问题，决定需要什么信息              │
└─────────────────────────────────────────────────────┘
                         ↓
┌─────────────────────────────────────────────────────┐
│  Action 1:   调用工具获取信息                        │
└─────────────────────────────────────────────────────┘
                         ↓
┌─────────────────────────────────────────────────────┐
│  Observation 1: 工具返回的结果                       │
└─────────────────────────────────────────────────────┘
                         ↓
                    循环或结束
                         ↓
┌─────────────────────────────────────────────────────┐
│  Final Answer: 综合所有信息给出答案                 │
└─────────────────────────────────────────────────────┘
```

## 16.4 ReAct Prompt 模板

In [ ]:
# 查看 ReAct 系统提示模板
REACT_TEMPLATE = """你是一个有用的助手，可以使用工具来回答问题。

可用工具:
{tools}

使用以下格式:

Thought: 你对问题的思考和分析
Action: 工具名称
Action Input: 工具参数 (JSON 格式)
Observation: 工具返回的结果
... (可以重复 Thought/Action/Observation 多次)
Thought: 我现在知道最终答案了
Final Answer: 最终答案

开始!

Question: {query}
Thought: """

print("=== ReAct Prompt 模板 ===\n")
print(REACT_TEMPLATE)

### Few-Shot 示例的重要性

In [ ]:
# 添加 Few-Shot 示例
FEW_SHOT_EXAMPLES = """
示例 1:
Question: 北京现在多少度？
Thought: 用户想知道北京的当前温度，我需要调用天气工具。
Action: get_weather
Action Input: {"city": "北京"}
Observation: 北京当前温度 22°C，晴天
Thought: 我已经获得了北京的温度信息。
Final Answer: 北京现在 22 摄氏度。

示例 2:
Question: 123 加 456 等于多少？
Thought: 这是一个计算问题，我需要使用计算器工具。
Action: calculator
Action Input: {"expression": "123 + 456"}
Observation: 579
Thought: 计算结果是 579。
Final Answer: 123 + 456 = 579。
"""

print("=== Few-Shot 示例 ===\n")
print(FEW_SHOT_EXAMPLES)

## 16.5 实战：实现简单的 ReAct Agent

In [ ]:
import re
from typing import Callable, Dict, Any

class SimpleReActAgent:
    """
    简化的 ReAct Agent 实现
    """
    
    def __init__(self, tools: Dict[str, Callable], llm_func: Callable):
        self.tools = tools
        self.llm_func = llm_func
        self.max_iterations = 5
    
    def _parse_action(self, text: str) -> tuple[str, str]:
        """
        解析 LLM 输出，提取 Action 和 Action Input
        """
        # 查找 Action:
        action_match = re.search(r'Action:\s*(\w+)', text)
        if not action_match:
            return None, None
        
        action = action_match.group(1)
        
        # 查找 Action Input:
        input_match = re.search(r'Action Input:\s*(\{.*?\})', text, re.DOTALL)
        if not input_match:
            return action, "{}"
        
        action_input = input_match.group(1)
        return action, action_input
    
    def _check_final_answer(self, text: str) -> tuple[bool, str]:
        """
        检查是否有最终答案
        """
        match = re.search(r'Final Answer:\s*(.*?)(?=\n|Thought:|Action:|$)', text, re.DOTALL)
        if match:
            return True, match.group(1).strip()
        return False, ""
    
    def _build_prompt(self, query: str, history: List[Dict] = None) -> str:
        """
        构建 ReAct Prompt
        """
        tools_desc = "\n".join([
            f"- {name}: {tool.__doc__ or tool.__name__}"
            for name, tool in self.tools.items()
        ])
        
        prompt = f"""你是一个可以使用工具的助手。

可用工具:
{tools_desc}

使用格式:
Thought: [你的思考]
Action: [工具名称]
Action Input: [JSON 格式的参数]
Observation: [工具返回的结果]
... (可重复)
Final Answer: [最终答案]

Question: {query}
Thought: """
        
        # 添加历史
        if history:
            for item in history:
                prompt += f"\n{item['role']}\n{item['content']}"
        
        return prompt
    
    async def run(self, query: str) -> Dict[str, Any]:
        """
        运行 ReAct 循环
        """
        history = []
        iterations = 0
        
        while iterations < self.max_iterations:
            # 构建提示
            prompt = self._build_prompt(query, history)
            
            # 调用 LLM
            response = await self.llm_func(prompt)
            
            # 检查是否有最终答案
            has_answer, answer = self._check_final_answer(response)
            if has_answer:
                return {
                    "answer": answer,
                    "iterations": iterations + 1,
                    "history": history
                }
            
            # 解析动作
            action, action_input = self._parse_action(response)
            
            if action is None:
                # 无法解析，让 LLM 重新生成
                history.append({"role": "assistant", "content": response})
                history.append({"role": "user", "content": "请使用正确的格式：Action: [工具名]"})
                iterations += 1
                continue
            
            # 执行工具
            if action not in self.tools:
                observation = f"错误: 未知工具 '{action}'"
            else:
                try:
                    params = json.loads(action_input)
                    result = await self.tools[action](**params)
                    observation = str(result)
                except Exception as e:
                    observation = f"错误: {str(e)}"
            
            # 记录历史
            history.append({"role": "assistant", "content": response})
            history.append({"role": "user", "content": f"Observation: {observation}"})
            
            iterations += 1
        
        return {
            "answer": "抱歉，我无法完成这个任务。",
            "iterations": iterations,
            "history": history
        }

# 打印 ReAct Agent 类定义
print("=== SimpleReActAgent 类已定义 ===\n")
print("主要方法:")
print("  - _parse_action(): 解析 LLM 输出中的 Action")
print("  - _check_final_answer(): 检查是否有最终答案")
print("  - _build_prompt(): 构建 ReAct Prompt")
print("  - run(): 运行 ReAct 循环")

## 16.6 ReAct 的优化技巧

### 1. Prompt 优化

| 技巧 | 说明 |
|------|------|
| **Few-Shot** | 提供完整的推理示例 |
| **清晰描述** | 详细说明每个工具的功能和参数 |
| **格式强制** | 明确要求输出格式 |
| **错误处理** | 告诉 LLM 如何处理错误 |

### 2. 解析优化

- 使用 **Structured Output** (如果模型支持)
- 使用 **Function Calling API** 直接获取结构化调用
- 多种解析策略（正则 + JSON 解析）

### 3. 执行优化

- **并行执行**：多个独立工具同时调用
- **缓存**：相同参数的调用返回缓存结果
- **超时控制**：防止工具调用无限等待
- **重试机制**：失败的工具自动重试

## 16.7 常见面试问题

**Q: ReAct 和 Chain-of-Thought 有什么区别？**
- A:
  | 特性 | CoT | ReAct |
  |------|-----|------|
  | 动作 | 仅思考 | 思考 + 行动 |
  | 信息源 | 仅依赖内部知识 | 可获取外部信息 |
  | 适用场景 | 推理任务 | 工具使用任务 |
  | 输出 | 直接答案 | 过程 + 答案 |

**Q: ReAct 陷入循环怎么办？**
- A:
  1. 设置最大迭代次数
  2. 检测重复的动作
  3. 添加「停止条件」检查
  4. 在 Prompt 中提示「如果无法解决，给出最佳猜测」

**Q: Function Calling 和 ReAct 的关系？**
- A:
  - Function Calling 是 ReAct 的一种实现方式
  - 传统 ReAct：解析文本获取 Action
  - Function Calling：LLM 直接返回结构化的函数调用
  - Function Calling 更可靠，但需要模型支持

**Q: 如何评估 ReAct Agent 的效果？**
- A:
  1. **成功率**：任务完成的比例
  2. **平均步数**：完成任务的迭代次数
  3. **工具使用准确率**：正确调用工具的比例
  4. **Token 消耗**：每次任务使用的 Token 数
  5. **人工评估**：答案质量和推理合理性

---

## 总结

本课程学习了 ReAct 模式的核心知识：

| 知识点 | 说明 |
|--------|------|
| **ReAct 模式** | Reasoning + Acting 循环 |
| **核心组件** | Thought → Action → Observation |
| **Prompt 模板** | 清晰的格式说明 + Few-Shot 示例 |
| **解析技巧** | 正则提取 / JSON 解析 / Function Calling |
| **优化方法** | Prompt 优化、并行执行、缓存 |
| **评估指标** | 成功率、步数、准确率、Token 消耗 |

**下一步**: L17 将学习对话 Agent 的实现！